In [1]:
## Imports
import GPy
import numpy as np
import pandas as pd
from scipy.linalg import  solve
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from gibbs_sampling import (
    BimodalSampleMultivariate,
    GaussianSampleMultivariate,
    HeavisideSampleMultivariate,
    SampleMultivariate,
    StudenttSampleMultivariate,
)

## Whether to reoptimise hyperparameters
OPTIMIZING = False
## Manually change the seed to create different train/test sets
seed = 42


In [2]:
## Data
df = pd.read_excel("./Concrete_Data.xls")
df_28d = df[df["Age (day)"] == 28]
df_28d = df_28d.drop(columns=["Age (day)"])
df_train, df_test = train_test_split(df_28d, test_size=0.25, random_state=seed)

X_train = df_train[
    [
        col
        for col in df_train.columns
        if col != "Concrete compressive strength(MPa, megapascals) "
    ]
]
y_train = df_train["Concrete compressive strength(MPa, megapascals) "]
X_test = df_test[X_train.columns]
y_test = df_test["Concrete compressive strength(MPa, megapascals) "]

X_scaler, y_scaler, unc_scaler = (
    StandardScaler(),
    StandardScaler(),
    StandardScaler(with_mean=False),
)
X_train = X_scaler.fit_transform(X_train.to_numpy())
y_scaler.fit(y_train.to_numpy().reshape(-1, 1))
unc_scaler.fit(y_train.to_numpy().reshape(-1, 1))
y_train = y_scaler.transform(y_train.to_numpy().reshape(-1, 1))
X_test = X_scaler.transform(X_test.to_numpy())

NOISE_LEVEL = 0.001

In [3]:
## Helper functions

class GridSearchHyperparameters:
    def __init__(
        self,
        model_hyperparameters: dict[str, list],
        kernel_hyperparameters: dict[str, list],
        model: GPy.Model,
        kernel: GPy.kern.Kern = GPy.kern.RBF,
        # noise_level: float = 0.5,
        noise_level: float = 0.0,
    ) -> None:
        self.model_hyperparameters = model_hyperparameters
        self.kernel_hyperparameters = kernel_hyperparameters
        self.model = model
        self.kernel = kernel
        self.noise_level = noise_level
        self.best_loglikelihood = -np.inf
        self.best_hyperparameters = None
        self.best_model = None

        self.model_hyp_names = list(self.model_hyperparameters.keys())
        self.kernel_hyp_names = list(self.kernel_hyperparameters.keys())

    def _generate_search_grid(self):
        from itertools import product

        return product(
            *[
                *self.model_hyperparameters.values(),
                *self.kernel_hyperparameters.values(),
            ]
        )

    def _initialise_model(self, hyperparameter_set: list) -> GPy.Model:
        model_hyps = hyperparameter_set[: len(self.model_hyperparameters)]
        kernel_hyps = hyperparameter_set[len(self.model_hyperparameters) :]
        return self.model(
            self.X,
            self.y,
            normalizer=None,
            noise_var=self.noise_level,
            kernel=self.kernel(
                self.X.shape[1],
                **{name: val for name, val in zip(self.kernel_hyp_names, kernel_hyps)},
            ),
            **{name: val for name, val in zip(self.model_hyp_names, model_hyps)},
        )

    def search(self) -> None:
        grid = self._generate_search_grid()
        for hyperparameter_set in grid:
            this_model = self._initialise_model(list(hyperparameter_set))
            this_model.optimize()
            log_likelihood = this_model.log_likelihood()
            if log_likelihood > self.best_loglikelihood:
                self.best_loglikelihood = log_likelihood
                self.best_hyperparameters = hyperparameter_set
                self.best_model = this_model

    def get_best_model(
        self, X: np.ndarray, y: np.ndarray
    ) -> tuple[GPy.Model, float, dict]:
        self.X = X
        self.y = y
        self.search()
        return (
            self.best_model,
            self.best_loglikelihood,
            {
                key: val
                for key, val in zip(
                    self.model_hyp_names + self.kernel_hyp_names,
                    self.best_hyperparameters,
                )
            },
        )


def kernel(data_dims, length_scale, variance, noise_scale):
    return GPy.kern.RBF(
        input_dim=data_dims, lengthscale=length_scale, variance=variance
    ) + GPy.kern.White(input_dim=data_dims, variance=noise_scale**2)


def generate_exp_unc_given_kernel(
    kernel,
    training_X: np.ndarray,
    testing_X: np.ndarray,
    training_y: np.ndarray,
    # noise_level: float = 1.0,
    noise_level: float = 0.2,
):
    Sigma_yy = kernel(training_X) + np.diag([noise_level] * training_X.shape[0])
    Sigma_ff = kernel(testing_X)
    Sigma_fy = kernel(testing_X, training_X)

    Sigma_fy_dot_inv_Sigma_yy_dot_y2 = Sigma_fy @ solve(
        Sigma_yy, training_y, assume_a="pos"
    )
    schur_Sigma_yy2 = Sigma_ff - Sigma_fy @ solve(Sigma_yy, Sigma_fy.T, assume_a="pos")
    y_squared2 = (
        training_y.reshape(1, -1) @ solve(Sigma_yy, training_y, assume_a="pos")
    ).flatten()[0]

    distribution_expectation = Sigma_fy_dot_inv_Sigma_yy_dot_y2.copy()
    distribution_covariance = schur_Sigma_yy2.copy()
    distribution_covariance += np.diag([noise_level] * testing_X.shape[0])

    return distribution_expectation, distribution_covariance, y_squared2

from itertools import combinations
from scipy.stats import t

def top_tier_from_summary(means, stds, ns, method_names=None, alpha=0.05):
    """
    Determine top-tier methods from mean ± std ± n summary statistics.
    
    Parameters:
    - means: array-like of means for each method
    - stds: array-like of standard deviations
    - ns: array-like of sample sizes
    - method_names: list of method names (default: Method_1, Method_2, ...)
    - alpha: significance level for pairwise comparisons
    
    Returns:
    - DataFrame with mean ± CI
    - List of top-tier methods
    """
    means = np.array(means)
    stds  = np.array(stds)
    ns    = np.array(ns)
    n_methods = len(means)
    
    if method_names is None:
        method_names = [f"Method_{i+1}" for i in range(n_methods)]
    
    # Standard errors and 95% CI
    sems = stds / np.sqrt(ns)
    ci95 = 1.96 * sems
    formatted = [f"{m:.2f}({ci:.2f})" for m, ci in zip(means, ci95)]
    
    summary_df = pd.DataFrame({
        "Method": method_names,
        "Mean ± 95% CI": formatted,
        "Mean": means
    }).sort_values("Mean", ascending=False)
    
    # Pairwise approximate t-tests (independent)
    pairs = []
    p_values = []
    for i, j in combinations(range(n_methods), 2):
        diff = means[i] - means[j]
        se_diff = np.sqrt(sems[i]**2 + sems[j]**2)
        t_stat = diff / se_diff
        df = ns[i] + ns[j] - 2
        p_val = 2 * t.sf(abs(t_stat), df)
        pairs.append((method_names[i], method_names[j]))
        p_values.append(p_val)
    
    # Build pairwise significance table
    pairwise_df = pd.DataFrame({
        "Method 1": [p[0] for p in pairs],
        "Method 2": [p[1] for p in pairs],
        "p-value": p_values,
        "Significant": [p < alpha for p in p_values]
    })
    
    # Determine top-tier (not significantly worse than best mean)
    best_method = summary_df.iloc[0]["Method"]
    top_tier = [best_method]
    for method in summary_df["Method"]:
        if method == best_method:
            continue
        mask = ((pairwise_df["Method 1"] == best_method) & (pairwise_df["Method 2"] == method)) | \
               ((pairwise_df["Method 2"] == best_method) & (pairwise_df["Method 1"] == method))
        significant = pairwise_df.loc[mask, "Significant"].values[0]
        if not significant:
            top_tier.append(method)
    
    return summary_df, pairwise_df, top_tier

In [4]:
## Parameters
if OPTIMIZING:
    gaussian_searcher = GridSearchHyperparameters(
        {},
        {
            "length_scale": [0.1, 0.5, 1, 5, 10],
            "variance": [0.1, 0.5, 1, 5, 10],
            "noise_scale": [0.01, 0.05, 0.1],
        },
        GPy.models.GPRegression,
        kernel=kernel,
        noise_level=NOISE_LEVEL,
    )
    opt_gaussian, gaussian_ll, gaussian_hyps = gaussian_searcher.get_best_model(
        X_train, y_train
    )
    print("Gaussian")
    print("loglikelihood", gaussian_ll)
    print(opt_gaussian)

    # Save the optimal parameters for later
    opt_gaussian_params = {
        "length_scale": opt_gaussian.kern.rbf.lengthscale.values[0],
        "variance": opt_gaussian.kern.rbf.variance.values[0],
        "noise_scale": opt_gaussian.kern.white.variance.values[0],
    }

    studentt_searcher = GridSearchHyperparameters(
        # {"deg_free": np.logspace(np.log10(0.1), np.log10(1000), num=100).tolist()},
        {"deg_free": np.logspace(np.log10(0.1), np.log10(1000), num=10).tolist()},
        {
            "length_scale": [0.1, 0.5, 1, 5, 10],
            "variance": [0.1, 0.5, 1, 5, 10],
            "noise_scale": [0.01, 0.05, 0.1],
        },
        GPy.models.TPRegression,
        kernel=kernel,
        noise_level=NOISE_LEVEL,
    )
    opt_studentt, studentt_ll, studentt_hyps = studentt_searcher.get_best_model(
        X_train, y_train
    )
    print("Student t")
    print("loglikelihood", studentt_ll)
    print(opt_studentt)

    # Save the optimal parameters for later
    opt_studentt_params = {
        "length_scale": opt_studentt.kern.rbf.lengthscale.values[0],
        "variance": opt_studentt.kern.rbf.variance.values[0],
        "noise_scale": opt_studentt.kern.white.variance.values[0],
        "nu": opt_studentt.nu.values[0],
    }

    bimodal_searcher = GridSearchHyperparameters(
        # {"n": np.logspace(np.log10(0.1), np.log10(1000), num=100).tolist()},
        {"n": np.logspace(np.log10(0.1), np.log10(1000), num=10).tolist()},
        {
            "length_scale": [0.1, 0.5, 1, 5, 10],
            "variance": [0.1, 0.5, 1, 5, 10],
            "noise_scale": [0.01, 0.05, 0.1],
        },
        GPy.models.BimodalRegression,
        kernel=kernel,
        noise_level=NOISE_LEVEL,
    )
    opt_bimodal, bimodal_ll, bimodal_hyps = bimodal_searcher.get_best_model(
        X_train, y_train
    )
    print("Bimodal")
    print("loglikelihood", bimodal_ll)
    print(opt_bimodal)

    # Save the optimal parameters for later
    opt_bimodal_params = {
        "length_scale": opt_bimodal.kern.rbf.lengthscale.values[0],
        "variance": opt_bimodal.kern.rbf.variance.values[0],
        "noise_scale": opt_bimodal.kern.white.variance.values[0],
        "n": opt_bimodal.nminusd.values[0],
    }

    heaviside_searcher = GridSearchHyperparameters(
        # {"n": np.logspace(np.log10(0.1), np.log10(1000), num=100).tolist()},
        {"n": np.logspace(np.log10(0.1), np.log10(1000), num=10).tolist()},
        {
            "length_scale": [0.1, 0.5, 1, 5, 10],
            "variance": [0.1, 0.5, 1, 5, 10],
            "noise_scale": [0.01, 0.05, 0.1],
        },
        GPy.models.HeavisideRegression,
        kernel=kernel,
        noise_level=NOISE_LEVEL,
    )
    opt_heaviside, heaviside_ll, heaviside_hyps = heaviside_searcher.get_best_model(
        X_train, y_train
    )
    print("heaviside")
    print("loglikelihood", heaviside_ll)
    print(opt_heaviside)

    # Save the optimal parameters for later
    opt_heaviside_params = {
        "length_scale": opt_heaviside.kern.rbf.lengthscale.values[0],
        "variance": opt_heaviside.kern.rbf.variance.values[0],
        "noise_scale": opt_heaviside.kern.white.variance.values[0],
        "n": opt_heaviside.nminusd.values[0],
    }
else:
    opt_gaussian_params = {
        "length_scale": 1.458349655669409,
        "variance": 1.3218770879121136,
        "noise_scale": 0.04208928402285215,
    }
    opt_studentt_params = {
        "length_scale": 1.458349614482665,
        "variance": 1.324520941260038,
        "noise_scale": 0.0421754599382221,
        "nu": 1000.0,
    }
    opt_bimodal_params = {
        "length_scale": 1.4586631169919522,
        "variance": 1.3141712109666979,
        "noise_scale": 0.041899508218700185,
        "n": 0.7060759899227766,
    }
    opt_heaviside_params = {
        "length_scale": 1.4582728826923144,
        "variance": 1.340543476028546,
        "noise_scale": 0.041470012994309496,
        "n": 4.647558813979918 ,
    }

In [5]:
## Distributions

gaussian_expectation, gaussian_covariance, gaussian_ysquared = (
    generate_exp_unc_given_kernel(
        kernel(
            X_train.shape[1],
            opt_gaussian_params["length_scale"],
            opt_gaussian_params["variance"],
            opt_gaussian_params["noise_scale"] ** 0.5,
        ).K,
        X_train,
        X_test,
        y_train,
        noise_level=NOISE_LEVEL,
    )
)

studentt_expectation, studentt_covariance, studentt_ysquared = (
    generate_exp_unc_given_kernel(
        kernel(
            X_train.shape[1],
            opt_studentt_params["length_scale"],
            opt_studentt_params["variance"],
            opt_studentt_params["noise_scale"] ** 0.5,
        ).K,
        X_train,
        X_test,
        y_train,
        noise_level=NOISE_LEVEL,
    )
)

bimodal_expectation, bimodal_covariance, bimodal_ysquared = (
    generate_exp_unc_given_kernel(
        kernel(
            X_train.shape[1],
            opt_bimodal_params["length_scale"],
            opt_bimodal_params["variance"],
            opt_bimodal_params["noise_scale"] ** 0.5,
        ).K,
        X_train,
        X_test,
        y_train,
        noise_level=NOISE_LEVEL,
    )
)

heaviside_expectation, heaviside_covariance, heaviside_ysquared = (
    generate_exp_unc_given_kernel(
        kernel(
            X_train.shape[1],
            opt_heaviside_params["length_scale"],
            opt_heaviside_params["variance"],
            opt_heaviside_params["noise_scale"] ** 0.5,
        ).K,
        X_train,
        X_test,
        y_train,
        noise_level=NOISE_LEVEL,
    )
)

list_of_distributions: dict[str, SampleMultivariate] = {
    "Gaussian": GaussianSampleMultivariate(
        gaussian_expectation, gaussian_covariance, noise_level=NOISE_LEVEL
    ),
    "Student \emph{t}": StudenttSampleMultivariate(
        studentt_expectation,
        studentt_covariance
        * (opt_studentt_params["nu"] + studentt_ysquared)
        / (opt_studentt_params["nu"] + len(X_train)),
        opt_studentt_params["nu"] + len(X_train),
        noise_level=NOISE_LEVEL,
    ),
    "Bimodal": BimodalSampleMultivariate(
        bimodal_expectation,
        bimodal_covariance,
        opt_bimodal_params["n"] + bimodal_ysquared - len(X_train),
        noise_level=NOISE_LEVEL,
    ),
    "Heaviside": HeavisideSampleMultivariate(
        heaviside_expectation,
        heaviside_covariance
        * (opt_heaviside_params["n"] +len(X_train)- heaviside_ysquared)
        / (opt_heaviside_params["n"]),
        opt_heaviside_params["n"],
        noise_level=NOISE_LEVEL,
    ),
}

In [6]:
## Performance

print(
    "Gaussian",
    r2_score(
        y_test,
        y_scaler.inverse_transform(
            list_of_distributions["Gaussian"].μ.reshape(-1, 1)
        ).flatten(),
    ),
)
print(
    "Student t",
    r2_score(
        y_test,
        y_scaler.inverse_transform(
            list_of_distributions["Student \emph{t}"].μ.reshape(-1, 1)
        ).flatten(),
    ),
)
print(
    "Bimodal",
    r2_score(
        y_test,
        y_scaler.inverse_transform(
            list_of_distributions["Bimodal"].μ.reshape(-1, 1)
        ).flatten(),
    ),
)
print(
    "Heaviside",
    r2_score(
        y_test,
        y_scaler.inverse_transform(
            list_of_distributions["Heaviside"].μ.reshape(-1, 1)
        ).flatten(),
    ),
)

Gaussian 0.7291613114340207
Student t 0.7291613053161976
Bimodal 0.7292113969733434
Heaviside 0.7288893929125028


In [7]:
## Run analysis
storage_df = pd.DataFrame(
    index=pd.MultiIndex.from_product(
        [list_of_distributions.keys(), [f"{i}" for i in range(len(y_test))]],
        names=["Distribution", "Test case"],
    ),
    dtype=float,
)
for i, y_test_value in enumerate(tqdm(y_test)):
    for name, distribution in list_of_distributions.items():
        mean_prediction = y_scaler.inverse_transform(
            distribution.μ[i].reshape(1, -1)
        ).flatten()[0]
        storage_df.loc[(name, f"{i}"), "true_value"] = y_test_value
        storage_df.loc[(name, f"{i}"), "predicted_value"] = mean_prediction
        for boundary_strength in [
            15,
            20,
            25,
            30,
            37,
            45,
            50,
            55,
            60,
            67,
            75,
        ]:
            storage_df.loc[(name, f"{i}"), f"prob_below_{boundary_strength}"] = (
                distribution._cdf(y_scaler.transform([[boundary_strength]]), i)
            ).flatten()[0]
            storage_df.loc[(name, f"{i}"), f"actual_below_{boundary_strength}"] = (
                y_test_value < boundary_strength
            )
storage_df.to_csv(f"concrete_predictions_{seed}.csv")

100%|██████████| 107/107 [00:02<00:00, 36.61it/s]


In [18]:
## Analyse
df = pd.read_csv(f"./concrete_predictions_{seed}.csv", index_col=[0, 1])

for percentage_level in [0.05]:
    for strength in [
        15,
        20,
        25,
        30,
        37,
        45,
        50,
        55,
        60,
        67,
        75,
    ]:
        means, stds, names = [],[],[]
        for distribution in df.index.get_level_values("Distribution").unique():
            names.append(distribution)
            this_data = df[
                (df.index.get_level_values("Distribution") == distribution)
            ]
            # true_values = df[
            #     (df.index.get_level_values("Distribution") == distribution)
            #     & (df[f"actual_below_{strength}"])
            # ]
            # false_values = df[
            #     (df.index.get_level_values("Distribution") == distribution)
            #     & (~df[f"actual_below_{strength}"])
            # ]
            # data = {
            #     'true': true_values[f"prob_below_{strength}"].values,
            #     'false': false_values[f"prob_below_{strength}"].values
            # }
            metrics = []
            for _ in range(1000):
                this_index = np.random.choice(this_data.index, size=len(this_data.index), replace=True)
                this_sample = this_data.loc[this_index]
                true_sample = this_sample[this_sample[f"actual_below_{strength}"]][f"prob_below_{strength}"].values
                false_sample = this_sample[~this_sample[f"actual_below_{strength}"]][f"prob_below_{strength}"].values
                tp = (true_sample >= percentage_level).sum()
                fn = (true_sample < percentage_level).sum()
                tn = (false_sample < percentage_level).sum()
                fp = (false_sample >= percentage_level).sum()
                misclassified = fn+fp

                # Compute F1 score
                f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else np.nan

                metrics.append([fp, fn,misclassified, f1])
            metrics = np.array(metrics)
            fp_mean, fn_mean,mc_mean, f1_mean = metrics.mean(axis=0)
            fp_std, fn_std,mc_std, f1_std = metrics.std(axis=0)
            means.append(f1_mean)
            stds.append(f1_std)
            print(
                f"Distribution: {distribution}, Strength: {strength}, Percentage Level: {percentage_level}"
            )
            print(
                f"          Misclassified: {mc_mean:.2f} ± {mc_std:.2f}"#, FPs: {fp_mean:.2f} ± {fp_std:.2f}, FNs: {fn_mean:.2f} ± {fn_std:.2f} "
                f"         F1 score: {f1_mean:.3f} ± {f1_std:.3f}, "
                # f"MCC: {((true_positives.sum() * true_negatives.sum()) - (false_positives.sum() * false_negatives.sum())) / math.sqrt((true_positives.sum() + false_positives.sum()) * (true_positives.sum() + false_negatives.sum()) * (true_negatives.sum() + false_positives.sum()) * (true_negatives.sum() + false_negatives.sum()))}"
            )
        _,_,top_tier = top_tier_from_summary(means, stds,[1000]*len(names),names)
        print(f'Strength: {strength}, Top-tier methods: {top_tier}')

Distribution: Gaussian, Strength: 15, Percentage Level: 0.05
          Misclassified: 14.68 ± 3.60         F1 score: 0.441 ± 0.124, 
Distribution: Student \emph{t}, Strength: 15, Percentage Level: 0.05
          Misclassified: 15.12 ± 3.55         F1 score: 0.434 ± 0.117, 
Distribution: Bimodal, Strength: 15, Percentage Level: 0.05
          Misclassified: 16.03 ± 3.70         F1 score: 0.415 ± 0.120, 
Distribution: Heaviside, Strength: 15, Percentage Level: 0.05
          Misclassified: 12.08 ± 3.14         F1 score: 0.489 ± 0.128, 
Strength: 15, Top-tier methods: ['Heaviside']
Distribution: Gaussian, Strength: 20, Percentage Level: 0.05
          Misclassified: 12.89 ± 3.47         F1 score: 0.677 ± 0.089, 
Distribution: Student \emph{t}, Strength: 20, Percentage Level: 0.05
          Misclassified: 13.08 ± 3.32         F1 score: 0.676 ± 0.084, 
Distribution: Bimodal, Strength: 20, Percentage Level: 0.05
          Misclassified: 20.00 ± 3.86         F1 score: 0.576 ± 0.085, 
Distribu